# DVC as “Git for data and models”

This notebook presents a minimal workflow for versioning data and model artifacts. T


## Installation

```bash
pip install dvc scikit-learn pandas joblib
```

In the project repository run:

```bash
git init
dvc init
```


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

Path("data").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

iris = load_iris(as_frame=True)
df = iris.frame
df.to_csv("data/iris.csv", index=False)
print(df.head())


## Adding data to DVC

In the terminal:

```bash
dvc add data/iris.csv
git add data/iris.csv.dvc data/.gitignore .dvc .dvcignore
git commit -m "Add Iris dataset tracked by DVC"
```

Optional local remote:

```bash
mkdir -p /tmp/dvc-storage
dvc remote add -d localstorage /tmp/dvc-storage
dvc push
```


In [ ]:
df = pd.read_csv("data/iris.csv")
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print({"accuracy": acc})

joblib.dump({"model": model, "accuracy": acc, "version": "v1"}, "models/iris_rf.joblib")


## Versioning the model

In the terminal:

```bash
dvc add models/iris_rf.joblib
git add models/iris_rf.joblib.dvc models/.gitignore
git commit -m "Add first RF model version"
dvc push
```

Then change model parameters, for example `n_estimators=200`, train the model again and run:

```bash
dvc add models/iris_rf.joblib
git add models/iris_rf.joblib.dvc
git commit -m "Update RF model version"
dvc push
```


## Summary
- Git stores code and lightweight DVC metadata.
- DVC stores large datasets and models in external storage.
- A model version without the matching data version is incomplete.
- We can go back to an old commit and run `dvc pull` to reproduce the exact data/model state.
